# These 4 upgrades are what separate a beginner Bronze layer from a production-level ingestion framework. I’ll explain each step-by-step, architecture level + code logic level.

We’ll cover:

- Incremental ingestion logic
- Auto Loader (streaming ingestion)
- File ingestion tracking
- Data quality logging

Think of this as **Bronze Layer v2** (Production Design).

# 1️⃣ Incremental Ingestion Logic (Very Important)

## ❓ Problem

If you run ingestion again:

- You don’t want to reload same files
- You only want new files / new data

This is called **Incremental Load**.

## 🧠 Two Ways Companies Do This

| Method          | How                                      |
|-----------------|------------------------------------------|
| **Timestamp**   | Load data where `ingestion_time > last_run` |
| **File Tracking**| Track processed file names             |
| **Auto Loader** | Automatically tracks files               |
| **Watermark**   | Streaming late data                     |

We’ll implement **File Tracking** + **Timestamp**.

# 🔹 Step 1 — Create Ingestion Log Table

This table tracks processed files.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS arijit_bronze_db.ingestion_log (
    table_name STRING,
    file_name STRING,
    ingestion_time TIMESTAMP,
    record_count INT
)
USING DELTA;

In [0]:
%sql
ALTER TABLE arijit_bronze_db.ingestion_log
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.minReaderVersion' = '2', 'delta.minWriterVersion' = '5');

# 🔹 Step 2 — Read Already Processed Files

In [0]:
processed_files = spark.sql("""
SELECT file_name FROM arijit_bronze_db.ingestion_log
WHERE table_name = 'users'
""")

In [0]:
# Convert to list:
processed_list = [row.file_name for row in processed_files.collect()]

In [0]:
processed_list

# 🔹 Step 3 — Read Only New Files

In [0]:
from pyspark.sql.functions import input_file_name

df = spark.read.csv("dbfs:/FileStore/raw/", header=True)
df = df.withColumn("source_file", input_file_name())

users_df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .option("badRecordsPath", "dbfs:/FileStore/tables/Arijit/badrecords/users/") \
            .load(users_path)

df_new = df.filter(~df.source_file.isin(processed_list))

In [0]:
# In the second notebook
view_name = dbutils.widgets.get("view_name")

# Now you can access the DataFrame using Spark SQL
df = spark.table(view_name)
df.show()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize Spark session
# spark = SparkSession.builder.master("local[*]").appName("NegationExample").getOrCreate()

# Sample data
data = [("file1.csv",), ("file2.csv",), ("file3.csv",), ("file4.csv",)]
columns = ["source_file"]

# Create a DataFrame
df = spark.createDataFrame(data, columns)

# Processed files list
processed_files = ["file1.csv", "file3.csv"]

# Filter using ~col("source_file").isin(processed_files)
filtered_df = df.filter(~col("source_file").isin(processed_files))

# Show the result
filtered_df.show()